In [1]:
from typing import Literal
from pydantic import BaseModel, Field
# 系统提示词
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.tools import tool

# 1.加载环境变量
load_dotenv()

# 2.初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# ==================方式三：基于BaseModel描述工具参数=================
# 2.1 定义参数描述的Pydantic模型类
class WeatherInput(BaseModel):
    location: str = Field(..., description='城市名称', min_length=1, max_length=10)
    units:Literal['Celsius','Fahrenheit'] = Field(
        default='Celsius',
        description='温度单位(摄氏度或华氏度)'
    )
    include_forecast:bool = Field(default=False,description='是否包含未来天气预报')

# 3.定义tool
@tool(args_schema=WeatherInput, description='获取指定位置当前时间的天气以及未来的天气(可选)')
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    # """
    # 获取指定位置当前时间的天气以及未来的天气(可选)
    # """
    # 上面description里写了 这里就不用写, 二者取其一
    temp = 22 if units == "celsius" else 72
    result = f"当前{location}的温度为:{temp} ° {units[0].upper()}"
    if include_forecast:
        result += "\n未来5天的天气: 晴天"
    return result


tools = [get_weather]

# 4.创建智能体
agent = create_agent(
    model=model,
    tools=tools,
)

# 5.流式调用
stream = agent.stream_events(
    {"messages": [HumanMessage(content="北京未来5天天气怎么样？用华氏温度")]},
    version="v3"
)

for message in stream.messages:
    for text in message.text:
        print(text, end="", flush=True)

D:\workspace\project\hm-insurance\agent-service\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
D:\workspace\project\hm-insurance\agent-service\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


我来帮您查询北京未来5天的天气情况。根据查询结果，北京未来5天的天气情况如下：

## 🌤️ 北京天气情况

- **当前温度**：72°F（华氏度）
- **未来5天天气**：**晴天** ☀️

接下来5天北京都是晴朗的好天气，非常适合外出活动。不过由于查询结果只显示了天气状况，建议您出行前关注具体的气温变化。

如果您想了解更详细的天气预报（比如每天的最高/最低温度等），欢迎随时告诉我！